In [1]:
from pathlib import Path
import sys
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))


In [2]:
import pickle, sys, warnings
import numpy as np
from scipy.signal import butter, sosfilt
from pylsl import StreamInlet, resolve_byprop

warnings.filterwarnings('ignore', message='X does not have valid feature names')
from preprocessing import extract_features

In [3]:
# ── Load model ────────────────────────────────────────────────────────────────
MODEL_PATH  = ROOT / 'notebooks/models/lgb_demo.pkl'
STREAM_NAME = 'PiEEG'

with open(MODEL_PATH, 'rb') as f:
    m = pickle.load(f)

model        = m['model']
W            = m['W'].astype(np.float64)
T_OPT        = int(m['T_OPT'])
MODEL_FS_EFF = int(m['FS_EFF'])
LP, HP       = float(m['LP']), float(m['HP'])
N_CH         = len(m['EEG_COLS'])
thresh       = float(m['thresh'])
USE_EA       = bool(m.get('USE_EA', True))  # default True for old models without the key

print(f'T_OPT={T_OPT}  MODEL_FS_EFF={MODEL_FS_EFF} Hz  LP={LP}  HP={HP}  N_CH={N_CH}')
print(f'thresh={thresh:.4f}  USE_EA={USE_EA}')
if USE_EA:
    print(f'W  shape={W.shape}  range=[{W.min():.3g}, {W.max():.3g}]  NaN={np.isnan(W).sum()}')

T_OPT=45  MODEL_FS_EFF=52 Hz  LP=0.5  HP=8.0  N_CH=16
thresh=0.5612  USE_EA=False


In [4]:
# ── Connect to LSL and infer sample rate ─────────────────────────────────────
print(f"Searching for '{STREAM_NAME}' ...", flush=True)
streams = resolve_byprop('name', STREAM_NAME, timeout=10)
assert streams, f"No stream '{STREAM_NAME}' found"
inlet = StreamInlet(streams[0])

nominal_fs = streams[0].nominal_srate()
STREAM_FS  = int(round(nominal_fs)) if nominal_fs > 0 else MODEL_FS_EFF
DECIM      = max(1, round(STREAM_FS / MODEL_FS_EFF))
FS_EFF     = STREAM_FS // DECIM

n_raw_needed = T_OPT * DECIM
print(f'Connected  ({streams[0].channel_count()} ch, {STREAM_FS} Hz)')
print(f'DECIM={DECIM}  FS_EFF={FS_EFF} Hz  (model expects {MODEL_FS_EFF} Hz)')
print(f'Need {n_raw_needed} raw samples to fill one window of {T_OPT} decimated samples')

Searching for 'PiEEG' ...
Connected  (17 ch, 250 Hz)
DECIM=5  FS_EFF=50 Hz  (model expects 52 Hz)
Need 225 raw samples to fill one window of 45 decimated samples


2026-06-11 19:20:30.454 (   0.535s) [          73DF4C]         api_config.cpp:126   INFO| Loaded default config
2026-06-11 19:20:30.456 (   0.538s) [          73DF4C]             common.cpp:78    INFO| git:64988c6a14b8dc3b3f270ece58eab4f480bfab43/branch:refs/tags/v1.17.7/build:Release/compiler:AppleClang-17.0.0.17000013/link:SHARED


In [5]:
# ── Collect exactly T_OPT decimated samples ───────────────────────────────────
nyq = STREAM_FS / 2.0
sos = butter(4, [LP / nyq, HP / nyq], btype='band', output='sos')
fz  = [np.zeros((sos.shape[0], 2)) for _ in range(N_CH)]

def bandpass_car(raw):
    out = np.empty(N_CH, dtype=np.float64)
    for c in range(N_CH):
        val   = float(raw[c]) if np.isfinite(raw[c]) else 0.0
        y, fz[c] = sosfilt(sos, [val], zi=fz[c])
        if not np.isfinite(fz[c]).all():
            fz[c] = np.zeros_like(fz[c])
        out[c] = y[0] if np.isfinite(y[0]) else 0.0
    out -= out.mean()   # CAR
    return out

print(f'Collecting {T_OPT} decimated samples ...', end='', flush=True)
dec_buf = []
n_raw   = 0
while len(dec_buf) < T_OPT:
    sample, _ = inlet.pull_sample()
    n_raw += 1
    x = bandpass_car(np.array(sample[:N_CH], dtype=np.float32))
    if n_raw % DECIM == 0:
        dec_buf.append(x)

X_raw = np.array(dec_buf, dtype=np.float64)   # (T_OPT, N_CH)
print(f' done ({n_raw} raw samples read)')
print(f'X_raw  shape={X_raw.shape}  NaN={np.isnan(X_raw).sum()}  Inf={np.isinf(X_raw).sum()}')
print(f'       range=[{X_raw.min():.4g}, {X_raw.max():.4g}]  std_mean={X_raw.std(0).mean():.4g}')

X_raw  shape=(45, 16)  NaN=0  Inf=0
       range=[-25.21, 35.24]  std_mean=8.353


In [6]:
# ── Preprocess: z-score → EA (if model used it) → feature extraction → predict

# 1. Trial z-score
X_z = (X_raw - X_raw.mean(0)) / (X_raw.std(0) + 1e-8)
print(f'After z-score:  NaN={np.isnan(X_z).sum()}  range=[{X_z.min():.3g}, {X_z.max():.3g}]')

# 2. Euclidean Alignment (only if model was trained with it)
if USE_EA:
    X_ea = np.nan_to_num(X_z @ W.T, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    print(f'After EA:       NaN={np.isnan(X_ea).sum()}  Inf={np.isinf(X_ea).sum()}  range=[{X_ea.min():.3g}, {X_ea.max():.3g}]')
else:
    X_ea = X_z.astype(np.float32)
    print('EA skipped (model trained without it)')

# 3. Feature extraction
feats = extract_features(X_ea, FS_EFF)
print(f'Features:       shape={feats.shape}  NaN={np.isnan(feats).sum()}  range=[{feats.min():.3g}, {feats.max():.3g}]')

# 4. Predict
score = float(model.predict_proba(feats[None, :])[0, 1])
print(f'\nScore = {score:.4f}  (thresh = {thresh:.4f})  →  {"PRESS DETECTED" if score >= thresh else "no detection"}')

After z-score:  NaN=0  range=[-3, 3.16]
EA skipped (model trained without it)
Features:       shape=(192,)  NaN=0  range=[-3, 2.12]

Score = 0.8394  (thresh = 0.5612)  →  PRESS DETECTED
